In [1]:
# Cai dat thu vien (neu chua co)
# pip install numpy scipy matplotlib

import numpy as np
from scipy.linalg import (
    expm, logm, sqrtm,
    schur, svd, norm
)

import time

# Thiet lap random seed de tai hien ket qua
np.random.seed(42)

# Machine epsilon
u = np.finfo(float).eps
print(f"Machine epsilon u = {u:.3e}")
print(f"NumPy version: {np.__version__}")

# Ham tao ma tran ngau nhien voi dieu kien so cho truoc
def random_matrix_cond(n, kappa):
    """Tao ma tran ngau nhien n x n voi condition number xap xi kappa."""

    U, _ = np.linalg.qr(np.random.randn(n, n))
    V, _ = np.linalg.qr(np.random.randn(n, n))

    s = np.logspace(0, -np.log10(kappa), n)

    return U @ np.diag(s) @ V.T

# Ham do sai so tuong doi Frobenius
def rel_err(F_approx, F_exact):
    denom = norm(F_exact, 'fro')

    return (
        norm(F_approx - F_exact, 'fro') / denom
        if denom > 0
        else float('inf')
    )

Machine epsilon u = 2.220e-16
NumPy version: 2.0.2


In [2]:
def test_matrices():
    """Tra ve dict cac ma tran thu nghiem chuan."""
    mats = {}

    # 1. Ma tran cheo hoa duoc, dieu kien tot
    mats['diagonal'] = np.diag([1.0, 2.0, 3.0, 4.0])

    # 2. Ma tran doi xung xac dinh duong (SPD)
    B = np.random.rand(4, 4)
    mats['SPD'] = B.T @ B + 4 * np.eye(4)

    # 3. Ma tran co tri rieng phuc (xoay)
    theta = np.pi / 4
    mats['rotation'] = np.array([
        [np.cos(theta), -np.sin(theta), 0, 0],
        [np.sin(theta),  np.cos(theta), 0, 0],
        [0, 0, np.cos(theta), -np.sin(theta)],
        [0, 0, np.sin(theta),  np.cos(theta)]
    ])

    # 4. Ma tran khong chinh tac (non-normal) - kha kho
    mats['non_normal'] = np.array([
        [-1.0, 500.0, 0.0, 0.0],
        [0.0, -1.0, 500.0, 0.0],
        [0.0, 0.0, -1.0, 500.0],
        [0.0, 0.0, 0.0, -1.0]
    ])

    # 5. Ma tran tri rieng lap (Jordan block 4x4)
    lam = 2.0
    mats['jordan'] = lam * np.eye(4) + np.diag([1, 1, 1], k=1)

    return mats


MATS = test_matrices()

for name, M in MATS.items():
    kappa = np.linalg.cond(M)
    eigvals = np.linalg.eigvals(M)

    print(
        f"{name:15s}: shape={M.shape}, "
        f"kappa={kappa:.2e}, "
        f"eigenvalues ~ {np.round(eigvals, 2)}"
    )

diagonal       : shape=(4, 4), kappa=4.00e+00, eigenvalues ~ [1. 2. 3. 4.]
SPD            : shape=(4, 4), kappa=2.04e+00, eigenvalues ~ [8.29 4.07 4.45 4.55]
rotation       : shape=(4, 4), kappa=1.00e+00, eigenvalues ~ [0.71+0.71j 0.71-0.71j 0.71+0.71j 0.71-0.71j]
non_normal     : shape=(4, 4), kappa=6.26e+10, eigenvalues ~ [-1. -1. -1. -1.]
jordan         : shape=(4, 4), kappa=2.27e+00, eigenvalues ~ [2. 2. 2. 2.]


In [3]:
def expm_eigenvector(A):
    """Tinh e^A qua phan tich tri rieng."""

    lam, X = np.linalg.eig(A)

    return X @ np.diag(np.exp(lam)) @ np.linalg.inv(X)


def expm_schur_parlett(A):
    """
    Tinh e^A qua Schur-Parlett (don gian hoa cho tri rieng phan biet).

    A = Q T Q^H  =>  e^A = Q e^T Q^H
    """

    T, Q = schur(A, output='complex')

    n = T.shape[0]

    F = np.zeros_like(T, dtype=complex)

    lam = np.diag(T)

    # Khoi tao duong cheo
    for i in range(n):
        F[i, i] = np.exp(lam[i])

    # Tinh cac sieu duong cheo theo thu tu p = 1, 2, ..., n-1
    for p in range(1, n):

        for i in range(n - p):

            j = i + p

            dl = lam[j] - lam[i]

            if abs(dl) > 1e-10:
                # Sai phan chia bac 1 (distinct eigenvalues)
                Fij = T[i, j] * (F[j, j] - F[i, i]) / dl

                # Dong gop tu cac phan tu trung gian (chi khi tri rieng phan biet)
                for k in range(i + 1, j):

                    dlk = lam[j] - lam[k]
                    dki = lam[k] - lam[i]

                    if abs(dlk) > 1e-10 and abs(dki) > 1e-10:

                        Fij += (
                            T[i, k] * F[k, j]
                            - F[i, k] * T[k, j]
                        ) / dl

            else:
                # Gioi han: f'(lambda) khi tri rieng trung nhau (confluent eigenvalues)
                # For this simplified version, if eigenvalues are repeated (dl approx 0),
                # we only handle the first super-diagonal term explicitly. Higher-order
                # terms for repeated eigenvalues are more complex and not covered by
                # this simplified approach to avoid NaNs.
                Fij = T[i, j] * np.exp(lam[i])

            F[i, j] = Fij

    return (Q @ F @ Q.conj().T).real


# ---------------------------------------------------------
# Chay thu nghiem tren cac ma tran chuan
# ---------------------------------------------------------

print(f"{'Ma tran':15s} | {'Eig err':10s} | {'Schur err':10s} "
      f"| {'t_eig (ms)':11s} | {'t_schur (ms)':12s}")

print("-" * 68)

for name, A in MATS.items():

    ref = expm(A)

    t0 = time.perf_counter()
    F_eig = expm_eigenvector(A).real
    t_eig = (time.perf_counter() - t0) * 1000

    t0 = time.perf_counter()
    F_sch = expm_schur_parlett(A)
    t_sch = (time.perf_counter() - t0) * 1000

    e_eig = rel_err(F_eig, ref)
    e_sch = rel_err(F_sch, ref)

    print(f"{name:15s} | {e_eig:10.2e} | {e_sch:10.2e} "
          f"| {t_eig:11.3f} | {t_sch:12.3f}")

Ma tran         | Eig err    | Schur err  | t_eig (ms)  | t_schur (ms)
--------------------------------------------------------------------
diagonal        |   0.00e+00 |   0.00e+00 |       4.076 |       32.583
SPD             |   1.84e-14 |   6.60e-15 |       0.350 |        0.981
rotation        |   7.74e-17 |   3.55e-16 |       6.011 |        0.394
non_normal      |   2.96e+31 |   1.00e+00 |       0.278 |        0.276
jordan          |   6.85e-01 |   2.65e-01 |       0.155 |        0.179


In [4]:
def expm_taylor(A, q):
    n = A.shape[0]
    F = np.eye(n); Ak = np.eye(n); fact = 1.0
    for k in range(1, q + 1):
        Ak = Ak @ A; fact *= k; F += Ak / fact
    return F

def expm_taylor_scaled(A, q, j=None):
    """Taylor voi scaling: tinh e^{A/2^j} roi binh phuong j lan."""
    norm_A = np.max(np.sum(np.abs(A), axis=1))
    if j is None:
        j = max(0, int(np.ceil(np.log2(norm_A))))
    As = A / (2 ** j)
    F = expm_taylor(As, q)
    for _ in range(j):
        F = F @ F
    return F

# Hàm tính sai số tương đối (để code chạy được)
def rel_err(F, ref):
    return np.linalg.norm(F - ref) / np.linalg.norm(ref)

# --- Script kiểm tra từ ảnh thứ hai ---

# Ma tran gay huy so nhat
A_bad = np.array([[-49.0, 24.0],
                  [-64.0, 31.0]])
ref = expm(A_bad)

print("Hien tuong huy so voi A = [[-49,24],[-64,31]]")
print(f" Ket qua chinh xac: \n{ref}\n")

print(f" {'Bac q':6s} | {'Taylor thuan (err)':20s} | {'Taylor + Scaling (err)':22s}")
print(" " + "-"*55)

for q in [5, 10, 15, 20, 30, 50, 59]:
    F_naive = expm_taylor(A_bad, q)
    F_scaled = expm_taylor_scaled(A_bad, q)

    e_naive = rel_err(F_naive, ref)
    e_scaled = rel_err(F_scaled, ref)

    print(f"{q:6d} | {e_naive:20.4e} | {e_scaled:22.4e}")

# Theo doi gia tri lon nhat cua cac so hang trung gian
print("\nGia tri lon nhat cua cac so hang Ak/k! (A_bad):")
n = A_bad.shape[0]
Ak = np.eye(n); fact = 1.0
for k in range(1, 21):
    Ak = Ak @ A_bad; fact *= k
    term_max = np.max(np.abs(Ak / fact))
    if k <= 5 or k % 5 == 0:
        print(f" k={k:2d}: max|A^k / k!| = {term_max:.3e}")

Hien tuong huy so voi A = [[-49,24],[-64,31]]
 Ket qua chinh xac: 
[[-0.73575876  0.5518191 ]
 [-1.4715176   1.10363824]]

 Bac q  | Taylor thuan (err)   | Taylor + Scaling (err)
 -------------------------------------------------------
     5 |           2.4580e+04 |             1.2558e-13
    10 |           9.3735e+05 |             7.2987e-14
    15 |           3.1139e+06 |             7.2987e-14
    20 |           2.0556e+06 |             7.2987e-14
    30 |           2.9958e+04 |             7.2987e-14
    50 |           7.4678e-04 |             7.2987e-14
    59 |           1.5528e-08 |             7.2987e-14

Gia tri lon nhat cua cac so hang Ak/k! (A_bad):
 k= 1: max|A^k / k!| = 6.400e+01
 k= 2: max|A^k / k!| = 5.760e+02
 k= 3: max|A^k / k!| = 3.275e+03
 k= 4: max|A^k / k!| = 1.392e+04
 k= 5: max|A^k / k!| = 4.733e+04
 k=10: max|A^k / k!| = 2.222e+06
 k=15: max|A^k / k!| = 8.756e+06
 k=20: max|A^k / k!| = 6.682e+06


In [5]:
def expm_eigenvector_general(A):
    lam, X = np.linalg.eig(A)
    return (X @ np.diag(np.exp(lam)) @ np.linalg.inv(X)).real

sizes = [4, 8, 16, 32, 64, 128, 256]
n_repeat = 5  # so lan lap de lay trung binh

print(f"{'n':>6} | {'t_eig (ms)':>12} | {'t_expm (ms)':>12} "
      f"| {'Speedup':>8} | {'err_eig':>10}")
print("-" * 60)

for n in sizes:
    # Tao ma tran ngau nhien xac dinh duong (luon co e^A xac dinh)
    B = np.random.randn(n, n)
    A = (B + B.T) / 2 - n * np.eye(n)  # doi xung, am dinh

    ref = expm(A)

    # Do thoi gian expm (SciPy -- Pade + Scaling&Squaring)
    t_expm_list = []
    for _ in range(n_repeat):
        t0 = time.perf_counter()
        F_expm = expm(A)
        t_expm_list.append((time.perf_counter() - t0) * 1000)
    t_expm = np.median(t_expm_list)

    # Do thoi gian phuong phap tri rieng (chi kha thi khi n nho)
    if n <= 64:
        t_eig_list = []
        for _ in range(n_repeat):
            t0 = time.perf_counter()
            F_eig = expm_eigenvector_general(A)
            t_eig_list.append((time.perf_counter() - t0) * 1000)
        t_eig = np.median(t_eig_list)
        err_eig = rel_err(F_eig, ref) # Ham rel_err ban da co tu file truoc
        speedup = t_eig / t_expm
        print(f"{n:>6} | {t_eig:12.3f} | {t_expm:12.3f} "
              f"| {speedup:>8.2f}x | {err_eig:10.2e}")
    else:
        print(f"{n:>6} | {'N/A (qua lon)':>12} | {t_expm:12.3f} "
              f"| {'---':>8} | {'---':>10}")

     n |   t_eig (ms) |  t_expm (ms) |  Speedup |    err_eig
------------------------------------------------------------
     4 |        0.091 |        5.303 |     0.02x |   2.77e-15
     8 |        0.212 |        3.035 |     0.07x |   3.43e-15
    16 |        0.822 |        2.766 |     0.30x |   2.38e-14
    32 |        2.581 |        2.528 |     1.02x |   1.53e-13
    64 |       10.346 |        3.979 |     2.60x |   1.35e-13
   128 | N/A (qua lon) |       72.437 |      --- |        ---
   256 | N/A (qua lon) |       72.441 |      --- |        ---


In [6]:
# Cài đặt lại các hàm lặp với theo dõi hội tụ
def matrix_sign_newton_track(A, max_iter=30, tol=1e-14):
    S = A.astype(complex).copy()
    residuals = []
    for k in range(max_iter):
        S_new = 0.5 * (S + np.linalg.inv(S))
        res = np.linalg.norm(S_new @ S_new - np.eye(S.shape[0]), 'fro')
        residuals.append(res)
        if res < tol:
            S = S_new
            break
        S = S_new
    return S, residuals

def matrix_sqrt_db_track(A, max_iter=30, tol=1e-14):
    X = A.astype(float).copy()
    Y = np.eye(A.shape[0])
    residuals = []
    for k in range(max_iter):
        X_inv = np.linalg.inv(X)
        Y_inv = np.linalg.inv(Y)
        X_new = 0.5 * (X + Y_inv)
        Y_new = 0.5 * (Y + X_inv)
        res = np.linalg.norm(X_new @ X_new - A, 'fro')
        residuals.append(res)
        if res < tol:
            X, Y = X_new, Y_new
            break
        X, Y = X_new, Y_new
    return X, Y, residuals

# ---------------------------------------------------------
# Ma trận thử nghiệm: SPD, để kiểm chứng
# ---------------------------------------------------------
B = np.array([[4.0, 2.0, 1.0],
              [2.0, 5.0, 3.0],
              [1.0, 3.0, 6.0]])

print("=" * 55)
print("MA TRAN B =")
print(B)
print(f"Tri rieng: {np.linalg.eigvals(B).real}")

# Thí nghiệm: Hàm dấu sign(B)
print("\n--- Hàm dau ma tran (Newton) ---")
# (Tiếp tục logic in ấn tùy theo nhu cầu của bạn)
S_B, res_sign = matrix_sign_newton_track(B)
ref_sign = np.diag(np.sign(np.linalg.eigvals(B).real))
# Luu y: sign(B) phu thuoc vao tri rieng thuc (deu duong => sign = I)
print(f"  sign(B) (tri rieng deu duong, nen sign(B) = I):")
print(f"  {np.diag(S_B.real)}")
print("  Residual ||S^2 - I||_F theo vong lap:")
for k, r in enumerate(res_sign):
    print(f"      Vong {k+1}: {r:.4e}")

# Thi nghiem: Can bac hai Denman-Beavers
print("\n--- Can bac hai ma tran (Denman-Beavers) ---")
X_sqrt, _, res_sqrt = matrix_sqrt_db_track(B)
ref_sqrt = sqrtm(B)
err_sqrt = np.linalg.norm(X_sqrt - ref_sqrt, 'fro') / np.linalg.norm(ref_sqrt, 'fro')
print(f"  Sai so so voi sqrtm(scipy): {err_sqrt:.2e}")
print("  Residual ||X^2 - B||_F theo vong lap:")
for k, r in enumerate(res_sqrt):
    print(f"      Vong {k+1}: {r:.4e}")

# Thi nghiem: Logarit -- kiem chung e^{logm(B)} = B
print("\n--- Logarit ma tran (SciPy logm) ---")
L_B   = logm(B)
check = expm(L_B)
err_log = np.linalg.norm(check - B, 'fro') / np.linalg.norm(B, 'fro')
print(f"  Sai so ||e^{{logm(B)}} - B||_F / ||B||_F = {err_log:.2e}")


# ---------------------------------------------------------
# Thi nghiem phan tich cuc voi SVD doi chieu
# ---------------------------------------------------------
print("\n--- Phan tich cuc A = U P ---")
A_rect = np.array([[1.0, 2.0, 0.0],
                   [0.0, 3.0, 4.0],
                   [5.0, 0.0, 1.0]])
Ua, Sa, Vat = np.linalg.svd(A_rect)
U_polar_ref = Ua @ Vat          # nhan tu truc giao tu SVD
P_polar_ref = Vat.T @ np.diag(Sa) @ Vat   # nhan tu xac dinh duong

# Newton lap cho nhan tu truc giao
X = A_rect.copy()
for k in range(30):
    X_new = 0.5 * (X + np.linalg.inv(X).T)
    if np.linalg.norm(X_new - X, 'fro') < 1e-14: break
    X = X_new
U_newton = X

err_polar = np.linalg.norm(U_newton - U_polar_ref, 'fro')
print(f"  ||U_Newton - U_SVD||_F = {err_polar:.2e}")

print(f"  Kiem tra truc giao: ||U^T U - I||_F = "
      f"{np.linalg.norm(U_newton.T @ U_newton - np.eye(3), 'fro'):.2e}")
print(f"  Kiem tra phan tich: ||A - UP||_F = "
      f"{np.linalg.norm(A_rect - U_newton @ P_polar_ref, 'fro'):.2e}")

MA TRAN B =
[[4. 2. 1.]
 [2. 5. 3.]
 [1. 3. 6.]]
Tri rieng: [9.34849393 3.73015912 1.92134694]

--- Hàm dau ma tran (Newton) ---
  sign(B) (tri rieng deu duong, nen sign(B) = I):
  [1. 1. 1.]
  Residual ||S^2 - I||_F theo vong lap:
      Vong 1: 2.1566e+01
      Vong 2: 5.1300e+00
      Vong 3: 1.0669e+00
      Vong 4: 1.3746e-01
      Vong 5: 4.1529e-03
      Vong 6: 4.2938e-06
      Vong 7: 4.6092e-12
      Vong 8: 4.1314e-24

--- Can bac hai ma tran (Denman-Beavers) ---
  Sai so so voi sqrtm(scipy): 8.74e-16
  Residual ||X^2 - B||_F theo vong lap:
      Vong 1: 1.7525e+01
      Vong 2: 2.8393e+00
      Vong 3: 1.6493e-01
      Vong 4: 7.1478e-04
      Vong 5: 1.3662e-08
      Vong 6: 2.2753e-15

--- Logarit ma tran (SciPy logm) ---
  Sai so ||e^{logm(B)} - B||_F / ||B||_F = 4.42e-15

--- Phan tich cuc A = U P ---
  ||U_Newton - U_SVD||_F = 9.24e-16
  Kiem tra truc giao: ||U^T U - I||_F = 2.30e-16
  Kiem tra phan tich: ||A - UP||_F = 3.77e-15
